In [1]:
!pip install transformers datasets accelerate wandb

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports


In [2]:
import wandb
from datasets import load_dataset,Dataset
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)


In [4]:
dataset = load_dataset("flytech/python-codes-25k", split="train")

# deduplicate first
df = dataset.to_pandas().drop_duplicates(subset="text")
dataset = Dataset.from_pandas(df, preserve_index=False)

print(len(dataset))
print(dataset.column_names)
print(dataset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

python-codes-25k.json:   0%|          | 0.00/26.4M [00:00<?, ?B/s]

python-codes-25k.jsonl:   0%|          | 0.00/25.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49626 [00:00<?, ? examples/s]

24813
['output', 'instruction', 'input', 'text']
{'output': "```python\ntasks = []\nwhile True:\n    task = input('Enter a task or type 'done' to finish: ')\n    if task == 'done': break\n    tasks.append(task)\nprint(f'Your to-do list for today: {tasks}')\n```", 'instruction': 'Help me set up my daily to-do list!', 'input': 'Setting up your daily to-do list...', 'text': "Help me set up my daily to-do list! Setting up your daily to-do list... ```python\ntasks = []\nwhile True:\n    task = input('Enter a task or type 'done' to finish: ')\n    if task == 'done': break\n    tasks.append(task)\nprint(f'Your to-do list for today: {tasks}')\n```"}


In [5]:
model_name = "FacebookAI/xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name,is_decoder=True) #force encoder to act as decoder (look only from one direction)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

XLMRobertaForCausalLM LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset.column_names
)

print(tokenized_dataset)

Map:   0%|          | 0/24813 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 24813
})


## Training

In [7]:
args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    num_train_epochs=2,
    learning_rate=5e-5,
    optim="adamw_torch",
    fp16=True,
    gradient_checkpointing=True,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    output_dir="/content/drive/MyDrive/roberta-fft",
    report_to="none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [19]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer
)

trainer.train(resume_from_checkpoint="/content/drive/MyDrive/roberta-fft/checkpoint-9500")

There were missing keys in the checkpoint model loaded: ['lm_head.decoder.weight', 'lm_head.decoder.bias', 'roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.Layer

Step,Training Loss
9550,0.401317
9600,0.482889
9650,0.395261
9700,0.406947
9750,0.467088
9800,0.454996
9850,0.441138
9900,0.395432
9950,0.470775
10000,0.443891


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12408, training_loss=0.10654268712093568, metrics={'train_runtime': 1802.4716, 'train_samples_per_second': 27.532, 'train_steps_per_second': 6.884, 'total_flos': 4.236610199014195e+16, 'train_loss': 0.10654268712093568, 'epoch': 2.0})

In [20]:
trainer.save_model("/content/drive/MyDrive/roberta-fft-final")
tokenizer.save_pretrained("/content/drive/MyDrive/roberta-fft-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/roberta-fft-final/tokenizer_config.json',
 '/content/drive/MyDrive/roberta-fft-final/tokenizer.json')

## Testing

In [6]:
import textwrap

model = AutoModelForCausalLM.from_pretrained("/content/drive/MyDrive/roberta-fft-final").to("cuda")
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/roberta-fft-final")

prompts = [
    "Write a Python function to sort a list",
    "Write a Python script to add two numbers",
]

for prompt in prompts:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    output_ids = model.generate(**inputs, max_new_tokens=100, repetition_penalty=1.2)
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    print("=" * 50)
    print(f"PROMPT:\n{prompt}")
    print("-" * 50)
    print("OUTPUT:")
    print(textwrap.fill(text, width=80))
    print("=" * 50)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

PROMPT:
Write a Python function to sort a list
--------------------------------------------------
OUTPUT:
Write a Python function to sort a list Execut Create Develop Fixing the code...
```py Design an array of integers in py Input Arrays = [2, 3] print(array) # Let
me run it for you I'm on top and weigh this! No sweats over Congratulations. We
have been given input data from your program using Pyglet library as well (i
like How many times), but there are no error messages here). The output is:
<modul
PROMPT:
Write a Python script to add two numbers
--------------------------------------------------
OUTPUT:
Write a Python script to add two numbers Create alphabetically Developing an
algorithm in py Implement Designed and implement the following I'm on it, hang
tight! Just give Input: ```python # Define functions for calculating all
elements of array. def find_all(x): if x <= 1 else None print (i) i = 2 while
lense is not present at index 3; j +=1); end=" " result[0] - start=0" } return